In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='water-quality-monitor-509313')

In [ ]:
print("hello")

hello


In [ ]:
ee.Initialize(project='water-quality-monitor-509313')

EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

In [ ]:
import ee
ee.Authenticate(force=True)


Successfully saved authorization token.


In [ ]:
import ee
ee.Initialize(project='water-quality-monitor-509313')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
import ee
import geemap

# Define the point of interest — Middle Fork Willamette River at Jasper, OR
point = ee.Geometry.Point([-122.9059126, 43.99818166])

# Get a Sentinel-2 image: filter by location, date range, and low cloud cover
collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(point)
    .filterDate("2023-06-01", "2023-09-30")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

image = collection.first()
print("Image date:", image.date().format("YYYY-MM-dd").getInfo())
print("Cloud %:", image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo())

Image date: 2023-07-19
Cloud %: 0


In [ ]:
# Compute NDWI = (Green - NIR) / (Green + NIR)
# Sentinel-2 bands: B3 = Green, B8 = NIR
ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")

# Set up a map centered on our point
Map = geemap.Map(center=[43.99818166, -122.9059126], zoom=13)

# Visualization parameters
rgb_vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
ndwi_vis = {"min": -1, "max": 1, "palette": ["brown", "white", "blue"]}

# Add layers: true-color image, then NDWI on top
Map.addLayer(image, rgb_vis, "True Color")
Map.addLayer(ndwi, ndwi_vis, "NDWI")
Map.addLayer(point, {}, "Station Location")

Map

Map(center=[43.99818166, -122.9059126], controls=(WidgetControl(options=['position', 'transparent_bg'], positi…

In [ ]:
ndwi_value = ndwi.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=10
).get("NDWI").getInfo()

print("NDWI at station point:", ndwi_value)

NDWI at station point: -0.06948109058927


In [ ]:
print(point.getInfo())

{'type': 'Point', 'coordinates': [-122.9059126, 43.99818166]}


In [ ]:
# Update these coordinates to a point you can see is clearly mid-water on the map
test_point = ee.Geometry.Point([-122.92, 44.005])  # adjust based on what you see

test_ndwi = ndwi.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=test_point,
    scale=10
).get("NDWI").getInfo()

print("NDWI at test point:", test_ndwi)

NDWI at test point: -0.6129032258064516


In [ ]:
stats = ndwi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=image.geometry(),
    scale=30,
    maxPixels=1e9
).getInfo()

print(stats)

{'NDWI_max': 0.7142857142857143, 'NDWI_min': -0.8853503184713376}


In [ ]:
Map = geemap.Map(center=[43.99818166, -122.9059126], zoom=13)

rgb_vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
ndwi_vis = {"min": -1, "max": 1, "palette": ["brown", "white", "blue"]}

# Create a proper binary water mask: NDWI > 0 = water, else transparent
water_mask = ndwi.gt(0).selfMask()

Map.addLayer(image, rgb_vis, "True Color")
Map.addLayer(water_mask, {"palette": ["blue"]}, "Water Mask (NDWI > 0)")
Map.addLayer(point, {}, "Station Location")

Map

Map(center=[43.99818166, -122.9059126], controls=(WidgetControl(options=['position', 'transparent_bg'], positi…

In [ ]:
# Second test location — Fox River near McHenry, IL
point2 = ee.Geometry.Point([-88.2334, 42.3386])  # approximate coords, adjust if needed

collection2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(point2)
    .filterDate("2023-06-01", "2023-09-30")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

image2 = collection2.first()
print("Image date:", image2.date().format("YYYY-MM-dd").getInfo())
print("Cloud %:", image2.get("CLOUDY_PIXEL_PERCENTAGE").getInfo())

ndwi2 = image2.normalizedDifference(["B3", "B8"]).rename("NDWI")

stats2 = ndwi2.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=image2.geometry(),
    scale=30,
    maxPixels=1e9
).getInfo()

print(stats2)

Image date: 2023-08-28
Cloud %: 1.783205
{'NDWI_max': 0.8117195004803074, 'NDWI_min': -0.9233511586452763}


In [ ]:
def get_water_mask(lat, lon, start_date, end_date, max_cloud_pct=10):
    """
    Fetches the clearest available Sentinel-2 image for a location/date range
    and returns the image, its NDWI band, and a binary water mask.
    """
    point = ee.Geometry.Point([lon, lat])  # note: EE wants [lon, lat]

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud_pct))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )

    image = collection.first()
    if image is None:
        raise ValueError("No suitable Sentinel-2 image found for this location/date range.")

    ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")
    water_mask = ndwi.gt(0).selfMask()

    return {
        "image": image,
        "ndwi": ndwi,
        "water_mask": water_mask,
        "point": point,
        "date": image.date().format("YYYY-MM-dd").getInfo(),
        "cloud_pct": image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo(),
    }


In [ ]:
result1 = get_water_mask(43.99818166, -122.9059126, "2023-06-01", "2023-09-30")
print(result1["date"], result1["cloud_pct"])

result2 = get_water_mask(42.3386, -88.2334, "2023-06-01", "2023-09-30")
print(result2["date"], result2["cloud_pct"])

2023-07-19 0
2023-08-28 1.783205
